In [1]:
!pip install opencv-python mediapipe numpy pulsectl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 29.2 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 9.5 MB/s eta 0:00:0010.4 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.8/87.8 MB 32.8 MB/s eta 0:00:00m eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 31.5 MB/s eta 0:00:00 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 30.2 MB/s eta 0:00:00 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 36.2 MB/s eta 0:00:000:00:010:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 40.9 MB/s eta 0:00:00
  Attempting uninstall: protobuf╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/16 [pyparsing]
    Found existing insta

In [1]:
import cv2
import mediapipe as mp
from math import hypot
import numpy as np
from pulsectl import Pulse, PulseVolumeInfo
 
cap = cv2.VideoCapture(2)
 
mpHands = mp.solutions.hands
hands = mpHands.Hands()
mpDraw = mp.solutions.drawing_utils
 
# Initialize PulseAudio control
pulse = Pulse('volume_control')  
while True:
    success, img = cap.read()
    if not success:
        print("Failed to capture image from camera.")
        break
 
    imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
 
    results = hands.process(imgRGB)
 
    lmList = []
    if results.multi_hand_landmarks:
        for handlandmark in results.multi_hand_landmarks:
            for id, lm in enumerate(handlandmark.landmark):
                h, w, _ = img.shape
                cx, cy = int(lm.x * w), int(lm.y * h)
                lmList.append([id, cx, cy])
            mpDraw.draw_landmarks(img, handlandmark, mpHands.HAND_CONNECTIONS)
 
    if lmList:
        x1, y1 = lmList[4][1], lmList[4][2]  # Thumb tip
        x2, y2 = lmList[8][1], lmList[8][2]  # Index finger tip
 
        cv2.circle(img, (x1, y1), 13, (255, 0, 0), cv2.FILLED)
        cv2.circle(img, (x2, y2), 13, (255, 0, 0), cv2.FILLED)
        cv2.line(img, (x1, y1), (x2, y2), (255, 0, 0), 3)
 
        length = hypot(x2 - x1, y2 - y1)
        
        # Scale length to a range suitable for volume (0 to 1)
        vol = np.interp(length, [30, 350], [0, 1])
 
        try:
            # Get current sink information
            sink_info = pulse.sink_list()[0] 
 
            # Set volume using PulseAudio
            pulse.volume_set_all_chans(sink_info, vol)
 
            # Debug print statements
            print(f"Length: {int(length)}")
            print(f"Setting volume to: {vol}")
 
            # Draw volume bar and percentage text
            volbar = np.interp(length, [30, 350], [400, 150])
            volper = np.interp(length, [30, 350], [0, 100])
            cv2.rectangle(img, (50, 150), (85, 400), (0, 0, 255), 4)
            cv2.rectangle(img, (50, int(volbar)), (85, 400), (0, 0, 255), cv2.FILLED)
            cv2.putText(img, f"{int(volper)}%", (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 98), 3)
 
        except Exception as e:
            print(f"Error setting volume: {e}")
 
    cv2.imshow('Hand Gesture Volume Control', img)
 
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
 
cap.release()
cv2.destroyAllWindows()

2025-07-25 09:22:31.991653: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-25 09:22:31.992154: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-25 09:22:31.994434: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-25 09:22:32.000897: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753406552.011299    4933 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753406552.01

Length: 275
Setting volume to: 0.7672995049647263
Length: 296
Setting volume to: 0.8335749919122206
Length: 329
Setting volume to: 0.936993246703562
Length: 341
Setting volume to: 0.9726078858080434
Length: 347
Setting volume to: 0.9925325441039731
Length: 346
Setting volume to: 0.988623874003341
Length: 345
Setting volume to: 0.9873825803295357
Length: 344
Setting volume to: 0.9834461578561261
Length: 342
Setting volume to: 0.9768669731164362
Length: 341
Setting volume to: 0.97466186171345
Length: 340
Setting volume to: 0.9705178307761632
Length: 342
Setting volume to: 0.9774824969398568
Length: 340
Setting volume to: 0.9693564445764592
Length: 336
Setting volume to: 0.9579459889269333
Length: 305
Setting volume to: 0.8598257131056768
Length: 203
Setting volume to: 0.5427688528237008
Length: 57
Setting volume to: 0.08522276329095442
Length: 21
Setting volume to: 0.0
Length: 161
Setting volume to: 0.4108479990299209
Length: 188
Setting volume to: 0.4947133059928206
Length: 208
Setting 